# Step 4c — Debate: A Society of Minds

The debate method asks: can a model do better by _arguing with itself_?
No search tool this time — debate is closed book by construction. Three
copies of the same model:

1. **Round 0** — each agent answers independently.
2. **Rounds 1–2** — each agent sees the _other_ agents' answers and
   reasoning, then answers again, revised or not.
3. **Verdict** — majority vote.

This is the "society of minds" setup ([Du et al. 2023](https://arxiv.org/abs/2305.14325)). Instead of calling
a prebuilt toolkit function first, this notebook **builds a one-round
debate from scratch**, piece by piece, so the machinery is transparent —
then shows the toolkit version that runs the real thing.

The building block is the **openai-agents SDK**. Two consequences of that
choice: the SDK targets OpenAI's Responses API, so only GPT models can
debate (Gemini sits this condition out), and it is async-native — Jupyter
already runs an event loop, so we `await` directly (IPython supports
top-level `await`; plain scripts use the sync wrappers instead).

Mind the meter: a full debate costs **9 LLM calls per question**
(3 agents × 3 rounds); this notebook's from-scratch walkthrough adds 6
more.

## 1. The building block: an Agent

An `Agent` is a role (our same answering system prompt), a model, and an
`output_type` (our same `Answer` schema, which the SDK validates).
`Runner.run` executes one turn. One setup line points the SDK at the same
API key the rest of the toolkit uses:


In [1]:
from agents import Agent, Runner, set_default_openai_client, set_tracing_disabled
from openai import AsyncOpenAI

from toolkit import prompts
from toolkit.answers import Answer
from toolkit.providers import PROVIDER_ENV, load_api_key
from toolkit.utils import load_jsonl

set_default_openai_client(AsyncOpenAI(api_key=load_api_key(PROVIDER_ENV["openai"])))

# By default, the SDK automatically records and sends agent run events—such as LLM generations,
# tool calls, and handoffs—to the OpenAI Traces dashboard. Setting this to True disables this behavior.
# This is useful to eliminate background latency, prevent error logs when using alternative LLM
# providers (like Gemini, Groq, or Azure OpenAI), or stop sensitive data from being uploaded to the cloud.
set_tracing_disabled(True)

MODEL = "gpt-5.4-mini-2026-03-17"
selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]
user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)

debater = Agent(
    name="Debater",
    model=MODEL,
    instructions=prompts.ANSWER_SYSTEM_PROMPT,
    output_type=Answer,
)

result = await Runner.run(debater, user_prompt)
result.final_output

Answer(answer_letter='A', confidence=0.78, reasoning='The study title suggests a parental-phone-usage survey focused on adults responsible for children, and this kind of study is typically conducted among U.S. parents and caregivers. The 600 participants are therefore most likely parents and caregivers in the United States.')

In [10]:
print(result.final_output.answer_letter)
print(result.final_output.confidence)
print(result.final_output.reasoning[:80], "...")

A
0.78
The study title suggests a parental-phone-usage survey focused on adults respons ...


In [3]:
result.__dict__

{'input': "QUESTION:\nWhat is the primary demographic of the 600 participants in the study titled, 'Mommy, do you love your phone more than me?'\n\nOPTIONS:\nA. Parents and caregivers in the United States\nB. Young people in the US ages 12 to 17\nC. Children under the age of 12\nD. Teens globally who use social media\n",
 'new_items': [MessageOutputItem(agent=Agent(name='Debater', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are an expert news-quiz contestant. Each question was written from a\nrecently published news article (within the last few weeks). You are NOT\ngiven the article — answer from what you know or can find.\n\nRules:\n- Pick the single best option: exactly one of A, B, C, or D.\n- Always commit to one letter, even if you are unsure.\n- Give 1-2 sentences of reasoning and a confidence between 0 and 1.\n', prompt=None, handoffs=[], model='gpt-5.4-mini-2026-03-17', model_settings=ModelSettings(temperature=None, top_p=None, frequency

One agent, one turn, one schema-validated `Answer` — exactly the
closed-book condition, rebuilt on the SDK.

## 2. Round 0 — three independent answers

A debate starts with independent opinions, so we run the _same_ agent
three times concurrently (`asyncio.gather`). Each run is a separate API
call with its own sampling, so the three "agents" can disagree — that
disagreement is the raw material of the debate:


In [4]:
import asyncio

N_AGENTS = 3

independent_runs = await asyncio.gather(
    *(Runner.run(debater, user_prompt) for _ in range(N_AGENTS))
)
round0 = [
    {
        "agent": agent_index,
        "answer_letter": run.final_output.answer_letter,
        "confidence": run.final_output.confidence,
        "reasoning": run.final_output.reasoning,
    }
    for agent_index, run in enumerate(independent_runs)
]

for entry in round0:
    print(
        f"agent {entry['agent']}: {entry['answer_letter']} ({entry['confidence']:.2f}) {entry['reasoning'][:80]}"
    )

agent 0: A (0.78) The study title suggests a survey of adult caregivers about phone use and parent
agent 1: A (0.90) The study title suggests it surveyed adults who are parenting, and the 600 parti
agent 2: A (0.96) The study described in that headline was a survey of U.S. parents and caregivers


## 3. A revision round

Now each agent gets to see the _other_ agents' answers — never its own —
and answer again. The revision prompt is just the question plus a "PEER
ANSWERS" block; here is exactly what agent 0 would read:


In [5]:
peers_of_agent0 = [entry for entry in round0 if entry["agent"] != 0]
print(
    prompts.build_debate_revision_prompt(
        question["question"], question["options"], peers_of_agent0
    )
)

QUESTION:
What is the primary demographic of the 600 participants in the study titled, 'Mommy, do you love your phone more than me?'

OPTIONS:
A. Parents and caregivers in the United States
B. Young people in the US ages 12 to 17
C. Children under the age of 12
D. Teens globally who use social media

PEER ANSWERS:
Agent 1 answered A (confidence 0.90): The study title suggests it surveyed adults who are parenting, and the 600 participants were described as U.S. parents/caregivers rather than children or teens. The other options don’t fit the likely respondent group.
Agent 2 answered A (confidence 0.96): The study described in that headline was a survey of U.S. parents and caregivers, not children or teens. The title suggests it examines parental phone use and its effects on family interactions.



The reviser is a second `Agent` with a different system prompt — same
commitment rules, but instructed to weigh the peers' arguments against
its own knowledge:


In [6]:
print(prompts.DEBATE_REVISION_SYSTEM_PROMPT)

You are an expert news-quiz contestant. Each question was written from a
recently published news article (within the last few weeks). You are NOT
given the article — answer from what you know.

Other contestants answered the same question; their answers and reasoning
are shown to you as additional advice. Weigh their arguments against your
own knowledge, then give your own (possibly revised) final answer.

Rules:
- Pick the single best option: exactly one of A, B, C, or D.
- Always commit to one letter, even if you are unsure.
- Give 1-2 sentences of reasoning and a confidence between 0 and 1.



In [7]:
reviser = Agent(
    name="Reviser",
    model=MODEL,
    instructions=prompts.DEBATE_REVISION_SYSTEM_PROMPT,
    output_type=Answer,
)

revision_turns = []
for agent_index in range(N_AGENTS):
    peer_answers = [entry for entry in round0 if entry["agent"] != agent_index]
    revision_turns.append(
        Runner.run(
            reviser,
            prompts.build_debate_revision_prompt(
                question["question"], question["options"], peer_answers
            ),
        )
    )
revised_runs = await asyncio.gather(*revision_turns)
round1 = [
    {
        "agent": agent_index,
        "answer_letter": run.final_output.answer_letter,
        "confidence": run.final_output.confidence,
        "reasoning": run.final_output.reasoning,
    }
    for agent_index, run in enumerate(revised_runs)
]

for before, after in zip(round0, round1):
    change_marker = (
        "" if before["answer_letter"] == after["answer_letter"] else "  <- changed"
    )
    print(
        f"agent {after['agent']}: {before['answer_letter']} -> "
        f"{after['answer_letter']} ({after['confidence']:.2f}){change_marker}"
    )

agent 0: A -> A (0.97)
agent 1: A -> A (0.95)
agent 2: A -> A (0.95)


## 4. The verdict — majority vote

The final round is aggregated deterministically, so the same transcript
always grades the same way: (1) most votes wins; (2) a tie goes to the
tied letter with the highest _mean_ confidence; (3) a residual tie goes
alphabetically:


In [8]:
from collections import Counter

votes = Counter(entry["answer_letter"] for entry in round1)
top_vote_count = max(votes.values())
leaders = sorted(
    letter for letter, vote_count in votes.items() if vote_count == top_vote_count
)
if len(leaders) > 1:

    def mean_confidence(letter):
        scores = [
            entry["confidence"] for entry in round1 if entry["answer_letter"] == letter
        ]
        return sum(scores) / len(scores)

    leaders.sort(key=lambda letter: (-mean_confidence(letter), letter))
verdict = leaders[0]

print(
    f"votes {dict(votes)} -> verdict {verdict} "
    f"(correct: {question['correct_letter']}) -> "
    f"{'CORRECT' if verdict == question['correct_letter'] else 'WRONG'}"
)

votes {'A': 3} -> verdict A (correct: B) -> WRONG


That's a complete one-round debate: independent answers → peer-informed
revision → deterministic vote. Everything else is bookkeeping.

## 5. The toolkit version

`debate_question_async()` runs the same loop with the tutorial's real
settings — `DEBATE_N_AGENTS = 3` agents and `DEBATE_N_ROUNDS = 2`
revision rounds, both in `toolkit/toolkit/config.py` — plus the pieces a
real experiment needs: per-turn retries on transient API errors, the full
transcript stored on the answer record, and the same
`_aggregate_votes()` tie-break we hand-rolled above. The final record's
confidence is the mean over the winning agents, and its reasoning comes
from the most confident winner:


In [9]:
from toolkit.debate import debate_question_async

debate = await debate_question_async(question, model=MODEL)

for round_number, round_entries in enumerate(debate["debate"]["transcript"]):
    label = "independent" if round_number == 0 else f"revision {round_number}"
    print(f"--- round {round_number} ({label}) ---")
    for entry in round_entries:
        print(
            f"  agent {entry['agent']}: {entry['answer_letter']} "
            f"({entry['confidence']:.2f}) {entry['reasoning'][:80]}"
        )
print(
    f"\nvotes {debate['debate']['vote_counts']} -> "
    f"final {debate['answer_letter']} "
    f"({'CORRECT' if debate['is_correct'] else 'WRONG'})"
)

--- round 0 (independent) ---
  agent 0: A (0.82) The study title suggests it was a survey of adults—specifically parents/caregive
  agent 1: A (0.87) The title suggests a study about parental phone use and its effects on family in
  agent 2: B (0.96) The study’s 600 participants were U.S. adolescents, specifically ages 12 to 17. 
--- round 1 (revision 1) ---
  agent 0: B (0.95) The study title points to youth perceptions of parental phone use, and the 600 p
  agent 1: B (0.94) The study surveyed 600 U.S. teens, ages 12 to 17, to understand their views on p
  agent 2: A (0.90) The study title points to adults being surveyed about phone use in parenting, no
--- round 2 (revision 2) ---
  agent 0: B (0.95) The study appears to survey 600 U.S. teens ages 12 to 17 about how parents’ phon
  agent 1: B (0.92) The study title suggests it examined how young people perceive parents' phone us
  agent 2: B (0.98) The study surveyed 600 U.S. teens, ages 12 to 17, about parents’ phone use. The 

vo

## 6. The full experiment, from the command line

Debating 100 questions is 900 calls, so the tutorial debates a single
contestant model. The merge script then folds every per-run answers file
— closed book, web search, and debate — into one tidy CSV:

```bash
# 04-2: debate, one model (900 calls — 9 per question)
uv run python scripts/04-2_generate_debate_answers.py \
    --model gpt-5.4-mini-2026-03-17 --parallel

# 04-3: merge every per-run file into one tidy CSV
uv run python scripts/04-3_combine_answers.py \
    --input-dir data/answers --glob 'answers_*.jsonl'
```


## 7. The map

| This notebook         | Where it lives                                                                                                                        |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| §1–3 agents & prompts | openai-agents SDK (`Agent`, `Runner.run`); `toolkit.prompts.DEBATE_REVISION_SYSTEM_PROMPT`, `build_debate_revision_prompt()`          |
| §4 the vote           | `toolkit.debate._aggregate_votes()`                                                                                                   |
| §5 one debate         | `toolkit.debate.debate_question_async()` (sync: `debate_question()`); knobs `DEBATE_N_AGENTS` / `DEBATE_N_ROUNDS` in `toolkit.config` |
| §6 at scale           | `scripts/04-2_generate_debate_answers.py`; `toolkit.debate.debate_questions()`; merged by `scripts/04-3_combine_answers.py`           |

---

### Next up 📊

Run the sweeps from notebooks 4a–4c, then open
[`04_answer_analysis.ipynb`](../analysis/04_answer_analysis.ipynb) for the
leaderboard: accuracy by method, what search buys, and what the debate
changed.
